# Thêm Thư Viện

In [2]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_dwh;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;')

## Đọc data từ SQL Server

In [5]:
query_phieumuon = """SELECT TOP (100) PMS.ID_phieu_muon, PMS.ID_ban_doc, PMS.ID_tai_lieu, Ma_xep_gia, Ngay_muon 
                        FROM oltp.Phieu_muon_sach PMS
                        JOIN olap.DIM_Ban_doc BD ON BD.ID_ban_doc = PMS.ID_ban_doc
                        JOIN olap.DIM_Xep_gia XG ON XG.ID_xep_gia = PMS.ID_xep_gia"""
df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)
print(df_phieumuon)

    ID_phieu_muon ID_ban_doc  ID_tai_lieu Ma_xep_gia  Ngay_muon
0               1          0        11089  GT0005338   20010219
1               2          0        11096  GT0066836   20010219
2               3          0        22194  GT0165834   20010219
3               4          0        14668  GT0066078   20010219
4               5          0        12495  GT0039796   20010219
..            ...        ...          ...        ...        ...
95             96          0          627  SKV011340   20020923
96             97          0          632  SKV011338   20020923
97             98          0          625  SKV011346   20020923
98             99          0          636  SKV011343   20020923
99            100          0          632  SKV011339   20020923

[100 rows x 5 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_9708\2320906618.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)


# Xử lý code

In [6]:
So_luot_muon = df_phieumuon.groupby(['ID_ban_doc', 'ID_tai_lieu', 'Ma_xep_gia', 'Ngay_muon'])['ID_phieu_muon'].count().reset_index()
So_luot_muon = So_luot_muon.rename(columns={'ID_phieu_muon': 'So_luot_muon'})
print(So_luot_muon)

   ID_ban_doc  ID_tai_lieu        Ma_xep_gia  Ngay_muon  So_luot_muon
0           0           31         SKV000138   20020917             5
1           0           41  (Không xác định)   20020827             1
2           0           55  (Không xác định)   20020826             1
3           0          173         SKV000165   20020910             1
4           0          188         SKV000158   20020910             2
..        ...          ...               ...        ...           ...
73   15146140        32994         GT0265887   20020101             1
74   98101150          295         SKV000804   20020909             1
75   98101150          356         SKV001684   20020909             5
76  N97105632         7553          ns000001   20010305             2
77  N97105632         7554  (Không xác định)   20010305             1

[78 rows x 5 columns]


## Load data

### [Nếu cần] Clear bảng

In [12]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.FACT_Muon"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [11]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.FACT_Muon (ID_ban_doc,
                                            ID_tai_lieu, 
                                            Ma_xep_gia, 
                                            ID_date, 
                                            So_luot_muon)
                VALUES (?, ?, ?, ?, ?)"""
for index, row in So_luot_muon.iterrows():
   # Trích xuất giá trị từ các cột
    values = (row['ID_ban_doc'],
              row['ID_tai_lieu'],
              row['Ma_xep_gia'],
              row['Ngay_muon'],
              row['So_luot_muon'])  # Nếu cột này có tên đúng
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()